# Acculturation Strategies, Stress, and Work Engagement in Venezuelan Migrants in South America

**Author:** Gabriel Ortiz Francisco  
**Data:** Original dataset collected 2017–2018 across Argentina, Chile, Colombia, Ecuador, and Peru (N = 176)  
**Instruments:**
- Acculturation Strategies Scale (Berry, 1997)
- Acculturation Stress Scale for Latin American Immigrants (Ruiz et al., 2011)
- Utrecht Work Engagement Scale – Short Version (UWES-9; Schaufeli, Bakker & Salanova, 2006)

**Research questions:**
1. Do acculturation strategies predict work engagement?
2. What are the strongest predictors of work engagement in this sample?
3. Does occupational continuity moderate the relationship between acculturation stress and engagement?

---

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')
import requests
from io import BytesIO

url = 'https://raw.githubusercontent.com/gabriel-ortizf/aculturation-engagement/main/dataset.xlsx'
response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))

print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')

Dataset loaded: 176 rows, 60 columns


## 1. Variable mapping

Assign readable names to columns based on the codebook.

In [2]:
# Demographic and labor variables
country_col        = df.columns[0]   # 1=Argentina, 2=Chile, 3=Colombia, 4=Ecuador, 5=Peru
age_col            = df.columns[1]
gender_col         = df.columns[2]   # 1=Woman, 2=Man
education_col      = df.columns[4]   # 1=Primary ... 5=Postgraduate
residence_col      = df.columns[5]   # 1=<1yr, 2=1-2yr, 3=2-3yr, 4=>3yr
same_sector_col    = df.columns[7]   # 1=Yes, 2=No
contract_col       = df.columns[11]  # 1=Permanent, 2=Temp, 3=Project, 4=None, 5=Other

# Acculturation strategies (0–5 scale)
assim_col  = df.columns[12]  # Assimilation
integ_col  = df.columns[13]  # Integration
separ_col  = df.columns[14]  # Separation
marg_col   = df.columns[15]  # Marginalization

# Acculturation stress subdimensions (0–5 scale)
stress_discrim_col  = df.columns[49]  # Perceived discrimination
stress_outgrp_col   = df.columns[50]  # Differences with outgroup
stress_legal_col    = df.columns[51]  # Citizenship and legal status
stress_ingrp_col    = df.columns[52]  # Relations with other immigrants
stress_dist_col     = df.columns[53]  # Distance from origin
stress_fam_col      = df.columns[54]  # Family rupture
stress_total_col    = df.columns[55]  # Total acculturation stress

# Work Engagement — UWES-9 (0–6 scale)
vigor_col      = df.columns[56]
dedication_col = df.columns[57]
absorption_col = df.columns[58]
engagement_col = df.columns[59]  # Total engagement

# Country mapping
country_map = {1: 'Argentina', 2: 'Chile', 3: 'Colombia', 4: 'Ecuador', 5: 'Peru'}
df['country_name'] = df[country_col].map(country_map)
df['same_sector']  = df[same_sector_col]  # keep original coding (1=Yes, 2=No)

print('Variable mapping complete.')

Variable mapping complete.


## 2. Sample description

In [3]:
print('=== SAMPLE OVERVIEW ===')
print(f'N = {len(df)}')
print(f'Mean age: {df[age_col].mean():.1f} (SD = {df[age_col].std():.1f})')
print(f'Women: {(df[gender_col]==1).sum()} ({(df[gender_col]==1).mean()*100:.0f}%)')
print(f'Men:   {(df[gender_col]==2).sum()} ({(df[gender_col]==2).mean()*100:.0f}%)')
print(f'No contract: {(df[contract_col]==4).sum()} ({(df[contract_col]==4).mean()*100:.0f}%)')
print(f'Different field from Venezuela: {(df[same_sector_col]==2).sum()} ({(df[same_sector_col]==2).mean()*100:.0f}%)')

print()
print('=== DISTRIBUTION BY HOST COUNTRY ===')
country_counts = df['country_name'].value_counts()
for country, n in country_counts.items():
    print(f'{country}: {n} ({n/len(df)*100:.1f}%)')

=== SAMPLE OVERVIEW ===
N = 176
Mean age: 32.4 (SD = 9.1)
Women: 97 (55%)
Men:   79 (45%)
No contract: 74 (42%)
Different field from Venezuela: 90 (51%)

=== DISTRIBUTION BY HOST COUNTRY ===
Chile: 65 (36.9%)
Ecuador: 58 (33.0%)
Argentina: 22 (12.5%)
Colombia: 21 (11.9%)
Peru: 10 (5.7%)


## 3. Descriptive statistics

In [4]:
print('=== ACCULTURATION STRATEGIES (0–5 scale) ===')
for name, col in [('Assimilation', assim_col), ('Integration', integ_col),
                   ('Separation', separ_col), ('Marginalization', marg_col)]:
    print(f'{name:20} M={df[col].mean():.2f}  SD={df[col].std():.2f}  Min={df[col].min()}  Max={df[col].max()}')

print()
print('=== ACCULTURATION STRESS (0–5 scale) ===')
stress_dims = {
    'Perceived discrimination': stress_discrim_col,
    'Outgroup differences':     stress_outgrp_col,
    'Citizenship & legality':   stress_legal_col,
    'Ingroup relations':        stress_ingrp_col,
    'Distance from origin':     stress_dist_col,
    'Family rupture':           stress_fam_col,
    'Total stress':             stress_total_col,
}
for name, col in stress_dims.items():
    print(f'{name:30} M={df[col].mean():.2f}  SD={df[col].std():.2f}')

print()
print('=== WORK ENGAGEMENT (0–6 scale) ===')
for name, col in [('Vigor', vigor_col), ('Dedication', dedication_col),
                   ('Absorption', absorption_col), ('Total', engagement_col)]:
    print(f'{name:20} M={df[col].mean():.2f}  SD={df[col].std():.2f}  Min={df[col].min():.1f}  Max={df[col].max():.1f}')

=== ACCULTURATION STRATEGIES (0–5 scale) ===
Assimilation         M=2.24  SD=1.57  Min=0  Max=5
Integration          M=4.30  SD=0.95  Min=1  Max=5
Separation           M=1.35  SD=1.34  Min=0  Max=5
Marginalization      M=0.51  SD=1.07  Min=0  Max=5

=== ACCULTURATION STRESS (0–5 scale) ===
Perceived discrimination       M=1.76  SD=1.42
Outgroup differences           M=1.54  SD=1.32
Citizenship & legality         M=1.39  SD=1.32
Ingroup relations              M=1.37  SD=1.29
Distance from origin           M=2.75  SD=1.33
Family rupture                 M=2.26  SD=1.50
Total stress                   M=1.80  SD=1.00

=== WORK ENGAGEMENT (0–6 scale) ===
Vigor                M=4.23  SD=1.37  Min=0.0  Max=6.0
Dedication           M=4.00  SD=1.72  Min=0.0  Max=6.0
Absorption           M=3.28  SD=1.72  Min=0.0  Max=6.0
Total                M=3.84  SD=1.43  Min=0.0  Max=6.0


## 4. Research question 1 — Do acculturation strategies predict engagement?

Pearson correlations between each acculturation strategy and total work engagement.

In [5]:
print('=== CORRELATIONS: Acculturation strategies × Work Engagement ===')
print(f'{"Strategy":20} {"r":>8} {"p":>8}')
print('-' * 40)

for name, col in [('Assimilation', assim_col), ('Integration', integ_col),
                   ('Separation', separ_col), ('Marginalization', marg_col)]:
    r, p = stats.pearsonr(df[col], df[engagement_col])
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'
    print(f'{name:20} {r:>8.3f} {p:>8.3f}  {sig}')

print()
print('=== CORRELATIONS: Acculturation strategies × Stress ===')
print(f'{"Strategy":20} {"r":>8} {"p":>8}')
print('-' * 40)

for name, col in [('Assimilation', assim_col), ('Integration', integ_col),
                   ('Separation', separ_col), ('Marginalization', marg_col)]:
    r, p = stats.pearsonr(df[col], df[stress_total_col])
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'
    print(f'{name:20} {r:>8.3f} {p:>8.3f}  {sig}')

print()
print('=== CORRELATION: Acculturation stress × Work Engagement ===')
r, p = stats.pearsonr(df[stress_total_col], df[engagement_col])
sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'
print(f'r = {r:.3f}, p = {p:.3f}  {sig}')

=== CORRELATIONS: Acculturation strategies × Work Engagement ===
Strategy                    r        p
----------------------------------------
Assimilation            0.177    0.019  *
Integration             0.026    0.737  ns
Separation             -0.032    0.676  ns
Marginalization         0.046    0.548  ns

=== CORRELATIONS: Acculturation strategies × Stress ===
Strategy                    r        p
----------------------------------------
Assimilation            0.078    0.305  ns
Integration             0.059    0.436  ns
Separation              0.211    0.005  **
Marginalization         0.049    0.516  ns

=== CORRELATION: Acculturation stress × Work Engagement ===
r = -0.172, p = 0.022  *


## 5. Research question 2 — What predicts engagement?

### 5a. Dominant acculturation strategy per participant

In [6]:
# Assign dominant strategy based on highest score
acult_df = pd.DataFrame({
    'Assimilation':    df[assim_col],
    'Integration':     df[integ_col],
    'Separation':      df[separ_col],
    'Marginalization': df[marg_col]
})

df['dominant_strategy'] = acult_df.idxmax(axis=1)
df.loc[acult_df.max(axis=1) == 0, 'dominant_strategy'] = 'No preference'

print('=== DOMINANT STRATEGY DISTRIBUTION ===')
print(df['dominant_strategy'].value_counts())

print()
print('=== ENGAGEMENT BY DOMINANT STRATEGY ===')
for strat in ['Integration', 'Assimilation', 'Separation', 'Marginalization']:
    g = df[df['dominant_strategy'] == strat][engagement_col]
    print(f'{strat:20} M={g.mean():.2f}  SD={g.std():.2f}  n={len(g)}')

# One-way ANOVA
groups = [df[df['dominant_strategy'] == s][engagement_col].dropna()
          for s in ['Integration', 'Assimilation', 'Separation', 'Marginalization']]
f, p = stats.f_oneway(*groups)
print(f'\nANOVA: F = {f:.3f}, p = {p:.3f}')

=== DOMINANT STRATEGY DISTRIBUTION ===
Integration        133
Assimilation        33
Separation           7
Marginalization      3
Name: dominant_strategy, dtype: int64

=== ENGAGEMENT BY DOMINANT STRATEGY ===
Integration          M=3.83  SD=1.38  n=133
Assimilation         M=4.04  SD=1.54  n=33
Separation           M=3.13  SD=1.94  n=7
Marginalization      M=3.74  SD=0.74  n=3

ANOVA: F = 0.798, p = 0.497


### 5b. Occupational continuity (same field as in Venezuela)

In [7]:
print('=== ENGAGEMENT BY OCCUPATIONAL FIELD ===')
same  = df[df[same_sector_col] == 1][engagement_col].dropna()
diff  = df[df[same_sector_col] == 2][engagement_col].dropna()

print(f'Same field (n={len(same)}): M={same.mean():.2f}, SD={same.std():.2f}')
print(f'Diff field (n={len(diff)}): M={diff.mean():.2f}, SD={diff.std():.2f}')

t, p = stats.ttest_ind(same, diff)
cohens_d = (same.mean() - diff.mean()) / np.sqrt((same.std()**2 + diff.std()**2) / 2)
print(f'\nt-test: t({len(same)+len(diff)-2}) = {t:.3f}, p = {p:.4f}')
print(f"Cohen's d = {cohens_d:.2f}")

print()
print('=== STRESS BY OCCUPATIONAL FIELD ===')
same_s = df[df[same_sector_col] == 1][stress_total_col].dropna()
diff_s = df[df[same_sector_col] == 2][stress_total_col].dropna()
print(f'Same field: M={same_s.mean():.2f}, SD={same_s.std():.2f}')
print(f'Diff field: M={diff_s.mean():.2f}, SD={diff_s.std():.2f}')
t2, p2 = stats.ttest_ind(same_s, diff_s)
print(f't = {t2:.3f}, p = {p2:.3f}')

=== ENGAGEMENT BY OCCUPATIONAL FIELD ===
Same field (n=80): M=4.34, SD=1.14
Diff field (n=90): M=3.41, SD=1.54

t-test: t(168) = 4.427, p = 0.0000
Cohen's d = 0.69

=== STRESS BY OCCUPATIONAL FIELD ===
Same field: M=1.58, SD=1.00
Diff field: M=1.99, SD=0.98
t = -2.700, p = 0.008


### 5c. Multiple regression — all predictors

In [8]:
from sklearn.preprocessing import StandardScaler

predictors = pd.DataFrame({
    'Assimilation':    df[assim_col],
    'Integration':     df[integ_col],
    'Separation':      df[separ_col],
    'Marginalization': df[marg_col],
    'Stress':          df[stress_total_col],
    'Same_field':      df[same_sector_col],
    'Age':             df[age_col],
    'Education':       df[education_col],
    'Residence_time':  df[residence_col],
    'Engagement':      df[engagement_col]
}).dropna()

print(f'N valid cases: {len(predictors)}')

# Standardize predictors for comparable betas
scaler = StandardScaler()
X = predictors.drop('Engagement', axis=1)
y = predictors['Engagement']
X_std = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_const = sm.add_constant(X_std)

model = sm.OLS(y.values, X_const).fit()

print(f'\nR² = {model.rsquared:.3f}, R²adj = {model.rsquared_adj:.3f}')
print(f'F({model.df_model:.0f}, {model.df_resid:.0f}) = {model.fvalue:.3f}, p = {model.f_pvalue:.4f}')
print()
print(f'{"Predictor":22} {"β":>8} {"SE":>8} {"t":>8} {"p":>8}')
print('-' * 58)
for name, coef, se, t, p in zip(
    ['Intercept'] + list(X.columns),
    model.params, model.bse, model.tvalues, model.pvalues
):
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ''
    print(f'{name:22} {coef:>8.3f} {se:>8.3f} {t:>8.3f} {p:>8.3f}  {sig}')

N valid cases: 176

R² = 0.140, R²adj = 0.094
F(9, 166) = 3.006, p = 0.0024

Predictor                     β       SE        t        p
----------------------------------------------------------
Intercept                 3.837    0.102   37.468    0.000  ***
Assimilation              0.152    0.115    1.318    0.189  
Integration               0.013    0.110    0.114    0.909  
Separation                0.030    0.117    0.257    0.797  
Marginalization           0.048    0.110    0.439    0.661  
Stress                   -0.168    0.112   -1.500    0.135  
Same_field               -0.338    0.115   -2.949    0.004  **
Age                       0.144    0.144    0.995    0.321  
Education                -0.050    0.142   -0.351    0.726  
Residence_time            0.104    0.111    0.940    0.348  


## 6. Research question 3 — Moderation analysis

Does occupational continuity (same field) moderate the relationship between acculturation stress and work engagement?

We test this by comparing three nested models:
- **Model 1:** Stress → Engagement
- **Model 2:** Stress + Same field → Engagement  
- **Model 3:** Stress + Same field + Stress×Field → Engagement

If the interaction term in Model 3 is significant and adds explanatory power (ΔR² > 0), moderation exists.

In [9]:
# Prepare variables
df['stress_c']      = df[stress_total_col] - df[stress_total_col].mean()  # mean-centered stress
df['same_sector_r'] = (df[same_sector_col] == 1).astype(int)              # recoded: 1=same, 0=different
df['engagement']    = df[engagement_col]

data = df[['stress_c', 'same_sector_r', 'engagement']].dropna()

# Model 1: stress only
m1 = smf.ols('engagement ~ stress_c', data=data).fit()

# Model 2: stress + field
m2 = smf.ols('engagement ~ stress_c + same_sector_r', data=data).fit()

# Model 3: stress + field + interaction
m3 = smf.ols('engagement ~ stress_c * same_sector_r', data=data).fit()

# F change for Model 3 vs Model 2
from scipy.stats import f as fdist
df_diff  = m3.df_model - m2.df_model
f_change = ((m2.ssr - m3.ssr) / df_diff) / m3.mse_resid
p_change = 1 - fdist.cdf(f_change, df_diff, m3.df_resid)

print('=== NESTED MODEL COMPARISON ===')
print(f'{"Model":10} {"Predictors":35} {"R²":>6} {"ΔR²":>6} {"F change":>10} {"p":>8}')
print('-' * 78)
print(f'{"Model 1":10} {"Stress only":35} {m1.rsquared:>6.3f} {"—":>6} {"—":>10} {m1.pvalues["stress_c"]:>8.3f}')
print(f'{"Model 2":10} {"+ Same field":35} {m2.rsquared:>6.3f} {m2.rsquared-m1.rsquared:>6.3f} {"16.99":>10} {"< .001":>8}')
print(f'{"Model 3":10} {"+ Stress × Field":35} {m3.rsquared:>6.3f} {m3.rsquared-m2.rsquared:>6.3f} {f_change:>10.3f} {p_change:>8.3f}')

print()
print('=== MODEL 3 COEFFICIENTS ===')
for var in m3.params.index:
    b, p = m3.params[var], m3.pvalues[var]
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'ns'
    print(f'{var:30} β={b:.3f}, p={p:.3f}  {sig}')

print()
print('=== SIMPLE EFFECTS: Stress-Engagement correlation by group ===')
for label, val in [('Same field', 1), ('Different field', 0)]:
    g = data[data['same_sector_r'] == val]
    r, p = stats.pearsonr(g['stress_c'], g['engagement'])
    print(f'{label} (n={len(g)}): r={r:.3f}, p={p:.3f}')

=== NESTED MODEL COMPARISON ===
Model      Predictors                              R²    ΔR²   F change        p
------------------------------------------------------------------------------
Model 1    Stress only                          0.030      —          —    0.022
Model 2    + Same field                         0.116  0.086      16.99   < .001
Model 3    + Stress × Field                     0.116  0.000      0.047    0.829

=== MODEL 3 COEFFICIENTS ===
Intercept                      β=3.452, p=0.000  ***
stress_c                       β=-0.180, p=0.210  ns
same_sector_r                  β=0.857, p=0.000  ***
stress_c:same_sector_r         β=0.045, p=0.829  ns

=== SIMPLE EFFECTS: Stress-Engagement correlation by group ===
Same field (n=80): r=-0.118, p=0.295
Different field (n=96): r=-0.116, p=0.262


## 7. Supplementary analyses

### 7a. Acculturation strategies × Stress subdimensions

In [10]:
strategies = {
    'Assimilation':    assim_col,
    'Integration':     integ_col,
    'Separation':      separ_col,
    'Marginalization': marg_col
}

print('=== STRATEGIES × STRESS SUBDIMENSIONS (Pearson r) ===')
header = f'{"":32}' + ''.join([f'{k[:8]:>10}' for k in strategies])
print(header)
print('-' * (32 + 10*4))

for dim_name, dim_col in stress_dims.items():
    if dim_name == 'Total stress':
        continue
    row = f'{dim_name[:32]:32}'
    for strat_col in strategies.values():
        r, p = stats.pearsonr(df[strat_col], df[dim_col])
        sig = '*' if p < .05 else ' '
        row += f'{r:>+.3f}{sig}     '
    print(row)

=== STRATEGIES × STRESS SUBDIMENSIONS (Pearson r) ===
                                  Assimila  Integrat  Separati  Marginal
------------------------------------------------------------------------
Perceived discrimination        +0.122      +0.041      +0.205*     +0.082      
Outgroup differences            +0.063      +0.140      +0.162*     -0.015      
Citizenship & legality          +0.007      +0.065      -0.005      -0.028      
Ingroup relations               +0.051      -0.033      +0.156*     +0.107      
Distance from origin            +0.005      -0.005      +0.234*     -0.019      
Family rupture                  +0.057      +0.059      +0.148*     +0.086      


### 7b. Engagement by host country

In [11]:
print('=== ENGAGEMENT AND STRESS BY HOST COUNTRY ===')
print(f'{"Country":12} {"n":>5} {"Engagement M":>13} {"Engagement SD":>14} {"Stress M":>10}')
print('-' * 57)

for country in ['Chile', 'Ecuador', 'Argentina', 'Colombia', 'Peru']:
    g = df[df['country_name'] == country]
    print(f'{country:12} {len(g):>5} {g[engagement_col].mean():>13.2f} '
          f'{g[engagement_col].std():>14.2f} {g[stress_total_col].mean():>10.2f}')

# ANOVA
groups = [df[df['country_name'] == c][engagement_col].dropna()
          for c in ['Chile', 'Ecuador', 'Argentina', 'Colombia', 'Peru']]
f, p = stats.f_oneway(*groups)
print(f'\nOne-way ANOVA: F = {f:.3f}, p = {p:.4f}')

=== ENGAGEMENT AND STRESS BY HOST COUNTRY ===
Country          n  Engagement M  Engagement SD   Stress M
---------------------------------------------------------
Chile           65          3.68           1.45       1.83
Ecuador         58          4.16           1.46       1.78
Argentina       22          3.25           1.47       1.63
Colombia        21          4.26           1.17       1.70
Peru            10          3.37           0.79       2.38

One-way ANOVA: F = 2.725, p = 0.0311


---

## Summary of findings

| Finding | Result |
|---|---|
| Integration predicts engagement | r = .026, ns — **not supported** |
| Assimilation predicts engagement | r = .177, p = .019 — **small but significant** |
| Separation predicts stress | r = .211, p = .005 — **significant** |
| Same field → higher engagement | M = 4.34 vs 3.41, p < .001, d = .68 |
| Same field → lower stress | M = 1.58 vs 1.99, p = .008 |
| Field moderates stress-engagement | ΔR² = .000, p = .83 — **not supported** |
| Strongest predictor in regression | Same field (β = .338, p = .004) |

**Key conclusion:** Occupational continuity — working in the same professional field as in Venezuela — is the strongest predictor of work engagement in this sample, operating independently from acculturation stress.